# Scorer Evaluation

Validates that a scorer reliably separates good plasmids from bad, making it a useful RL training signal.

**Approach:** Take real (prompt, sequence) pairs from training data, derive negative classes
(random DNA, shuffled, mismatched, truncated), score everything, and measure separation.

In [ ]:
from __future__ import annotations

import random
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

## Config

In [ ]:
DATA_DIR = ROOT / "data"
TRAINING_PAIRS_PATH = DATA_DIR / "training_pairs_sample.parquet"
MOTIF_REGISTRY_PATH = DATA_DIR / "motif_registry.parquet"

SCORER_NAME = "alignment"  # "alignment" or "motif"
SEED = 42
TRUNCATION_FRACTIONS = [0.25, 0.50, 0.75]

random.seed(SEED)
np.random.seed(SEED)

## Load data

In [ ]:
pairs_df = pd.read_parquet(TRAINING_PAIRS_PATH)

# Resolve column names — training pairs may use different conventions
prompt_col = "token_prompt" if "token_prompt" in pairs_df.columns else "prompt"
seq_col = "token_completion" if "token_completion" in pairs_df.columns else "sequence"

# If only full_text exists, split on <SEQ>
if prompt_col not in pairs_df.columns and "full_text" in pairs_df.columns:
    def _split_full_text(ft):
        for sep in ["<SEQ>", "<SEP>"]:
            if sep in ft:
                parts = ft.split(sep, 1)
                prompt = parts[0] + sep
                seq = parts[1].replace("<EOS>", "").strip()
                return prompt, seq
        return ft, ""
    splits = pairs_df["full_text"].apply(_split_full_text)
    pairs_df["prompt"] = splits.apply(lambda x: x[0])
    pairs_df["sequence"] = splits.apply(lambda x: x[1])
    prompt_col, seq_col = "prompt", "sequence"

prompts = pairs_df[prompt_col].tolist()
sequences = pairs_df[seq_col].tolist()

print(f"Loaded {len(prompts)} pairs")
print(f"Prompt example: {prompts[0][:120]}...")
print(f"Sequence length range: {min(len(s) for s in sequences):,} – {max(len(s) for s in sequences):,} bp")

## Initialize scorer

In [ ]:
from post_training.scorers import build_scorer

if SCORER_NAME == "alignment":
    scorer = build_scorer("alignment", motif_lookup_path=str(MOTIF_REGISTRY_PATH))
elif SCORER_NAME == "motif":
    scorer = build_scorer("motif", motif_db_path=str(MOTIF_REGISTRY_PATH))
else:
    raise ValueError(f"Unknown scorer: {SCORER_NAME}")

print(f"Scorer: {type(scorer).__name__}")

## Generate negative classes

In [ ]:
HARD_PREFIXES = {"AMR_", "ORI_", "PROM_", "REPORTER_", "REP_", "TAG_", "ELEM_"}


def random_dna(length: int) -> str:
    return "".join(random.choices("ATGC", k=length))


def dinucleotide_shuffle(seq: str) -> str:
    """Shuffle preserving dinucleotide frequencies (Altschul-Erickson)."""
    seq = seq.upper()
    dinucs = [seq[i:i+2] for i in range(0, len(seq) - 1, 2)]
    random.shuffle(dinucs)
    result = "".join(dinucs)
    if len(seq) % 2 == 1:
        result += seq[-1]
    return result


def drop_random_hard_tokens(prompt: str, drop_frac: float = 0.5) -> str:
    """Remove a fraction of hard tokens from the prompt."""
    tokens = re.findall(r"<[^>]+>", prompt)
    hard_tokens = [t for t in tokens if any(t.strip("<>").startswith(p) for p in HARD_PREFIXES)]
    n_drop = max(1, int(len(hard_tokens) * drop_frac))
    to_drop = set(random.sample(hard_tokens, min(n_drop, len(hard_tokens))))
    return re.sub(r"<[^>]+>", lambda m: "" if m.group() in to_drop else m.group(), prompt)


# Build all input classes
n = len(prompts)

# Mismatched: rotate sequences by half so every pair is wrong
shift = n // 2
mismatched_seqs = sequences[shift:] + sequences[:shift]

# Random DNA
random_seqs = [random_dna(len(s)) for s in sequences]

# Dinucleotide-shuffled
shuffled_seqs = [dinucleotide_shuffle(s) for s in sequences]

# Truncated at various fractions
truncated = {}
for frac in TRUNCATION_FRACTIONS:
    truncated[frac] = [s[:int(len(s) * frac)] for s in sequences]

# Partial prompts (fewer expected tokens, same sequence)
partial_prompts = [drop_random_hard_tokens(p) for p in prompts]

print("Negative classes generated")

## Score all classes

In [ ]:
from time import perf_counter


def score_class(name: str, p: list[str], s: list[str]) -> np.ndarray:
    t0 = perf_counter()
    scores = scorer.score_batch(p, s)
    elapsed = perf_counter() - t0
    arr = np.array(scores, dtype=np.float64)
    print(f"  {name:20s}  mean={arr.mean():.4f}  std={arr.std():.4f}  "
          f"med={np.median(arr):.4f}  [{arr.min():.4f}, {arr.max():.4f}]  "
          f"({elapsed:.1f}s)")
    return arr


print(f"Scoring {n} pairs per class with {type(scorer).__name__}...\n")

results = {}
results["real_matched"] = score_class("real_matched", prompts, sequences)
results["mismatched"] = score_class("mismatched", prompts, mismatched_seqs)
results["random_dna"] = score_class("random_dna", prompts, random_seqs)
results["shuffled"] = score_class("shuffled", prompts, shuffled_seqs)
for frac in TRUNCATION_FRACTIONS:
    results[f"truncated_{int(frac*100)}pct"] = score_class(
        f"truncated_{int(frac*100)}%", prompts, truncated[frac]
    )
results["partial_prompt"] = score_class("partial_prompt", partial_prompts, sequences)

## Separation metrics

In [ ]:
real = results["real_matched"]
negative_classes = [k for k in results if k != "real_matched" and "truncated" not in k and k != "partial_prompt"]

rows = []
for neg_name in negative_classes:
    neg = results[neg_name]
    gap = real.mean() - neg.mean()
    ratio = real.mean() / max(neg.mean(), 1e-8)

    # AUROC: real=1, negative=0
    labels = np.concatenate([np.ones(len(real)), np.zeros(len(neg))])
    scores_combined = np.concatenate([real, neg])
    try:
        auroc = roc_auc_score(labels, scores_combined)
    except ValueError:
        auroc = float("nan")

    # IQR overlap
    real_q1, real_q3 = np.percentile(real, [25, 75])
    neg_q1, neg_q3 = np.percentile(neg, [25, 75])
    overlap = max(0, min(real_q3, neg_q3) - max(real_q1, neg_q1))

    rows.append({
        "negative": neg_name,
        "mean_real": round(real.mean(), 4),
        "mean_neg": round(neg.mean(), 4),
        "gap": round(gap, 4),
        "ratio": round(ratio, 2),
        "auroc": round(auroc, 4),
        "iqr_overlap": round(overlap, 4),
    })

sep_df = pd.DataFrame(rows)
sep_df

## Score distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

plot_groups = [
    ("vs Random DNA", ["real_matched", "random_dna"]),
    ("vs Shuffled", ["real_matched", "shuffled"]),
    ("vs Mismatched", ["real_matched", "mismatched"]),
]

colors = {
    "real_matched": "#2ecc71",
    "random_dna": "#e74c3c",
    "shuffled": "#e67e22",
    "mismatched": "#9b59b6",
}

for ax, (title, keys) in zip(axes, plot_groups):
    for k in keys:
        ax.hist(results[k], bins=30, alpha=0.6, label=k.replace("_", " "),
                color=colors.get(k, "gray"), edgecolor="white", linewidth=0.5)
    ax.set_title(title)
    ax.set_xlabel("Score")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Count")
fig.suptitle(f"{type(scorer).__name__} — Score Distributions", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Graceful degradation (truncation)

In [ ]:
trunc_keys = sorted([k for k in results if "truncated" in k],
                     key=lambda k: int(k.split("_")[1].replace("pct", "")))

fracs = [int(k.split("_")[1].replace("pct", "")) / 100 for k in trunc_keys]
means = [results[k].mean() for k in trunc_keys]
stds = [results[k].std() for k in trunc_keys]

# Add the full-length point
fracs = fracs + [1.0]
means = means + [real.mean()]
stds = stds + [real.std()]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(fracs, means, yerr=stds, fmt="o-", capsize=4, color="#2c3e50", markersize=7)
ax.set_xlabel("Fraction of sequence retained")
ax.set_ylabel("Mean score")
ax.set_title(f"{type(scorer).__name__} — Score vs Truncation")
ax.set_xticks(fracs)
ax.set_xticklabels([f"{int(f*100)}%" for f in fracs])
fig.tight_layout()
plt.show()

# Check monotonicity
is_monotonic = all(means[i] <= means[i+1] for i in range(len(means) - 1))
print(f"Monotonically increasing with sequence length: {is_monotonic}")

## Per-component attribution (detailed scoring)

In [ ]:
N_DETAILED = min(10, len(prompts))

detailed_results = []
for i in range(N_DETAILED):
    detail = scorer.score_sequence_detailed(prompts[i], sequences[i])
    detail["idx"] = i
    detailed_results.append(detail)

# Show per-motif breakdown for first few
for d in detailed_results[:3]:
    print(f"\n── Pair {d['idx']} ─────────────────────────")
    print(f"  Reward: {d['reward']:.4f}")

    # AlignmentScorer format
    if "per_motif" in d:
        print(f"  Hard tokens: {d.get('n_hard_tokens', '?')}  Found: {d.get('n_found', '?')}  "
              f"QC pass rate: {d.get('qc_pass_rate', '?')}")
        for m in d["per_motif"]:
            status = "FOUND" if m["found"] else "MISS"
            print(f"    {m['token']:30s}  score_ratio={m['score_ratio']:.4f}  [{status}]")

    # MotifScorer format
    if "token_scores" in d:
        print(f"  Expected: {d.get('expected', '?')}  Found: {d.get('found', '?')}  "
              f"Recall: {d.get('recall', '?')}  Quality: {d.get('quality', '?')}")
        for tok, info in d["token_scores"].items():
            status = "FOUND" if info["found"] else "MISS"
            print(f"    {tok:30s}  score={info['score']:.4f}  "
                  f"pct_id={info['pct_id']}  cov={info['coverage']}  [{status}]")

## Attribution sanity check: real vs random

In [ ]:
# Compare detailed scores: real sequence vs random DNA for same prompt
for i in range(min(3, len(prompts))):
    real_detail = scorer.score_sequence_detailed(prompts[i], sequences[i])
    rand_detail = scorer.score_sequence_detailed(prompts[i], random_seqs[i])

    print(f"\n── Pair {i} ─────────────────────────")
    print(f"  Real reward:   {real_detail['reward']:.4f}")
    print(f"  Random reward: {rand_detail['reward']:.4f}")

    if "per_motif" in real_detail:
        real_motifs = {m["token"]: m["score_ratio"] for m in real_detail["per_motif"]}
        rand_motifs = {m["token"]: m["score_ratio"] for m in rand_detail.get("per_motif", [])}
        for tok in real_motifs:
            r_real = real_motifs.get(tok, 0)
            r_rand = rand_motifs.get(tok, 0)
            print(f"    {tok:30s}  real={r_real:.4f}  random={r_rand:.4f}")

    if "token_scores" in real_detail:
        for tok in real_detail["token_scores"]:
            r_real = real_detail["token_scores"][tok]["score"]
            r_rand = rand_detail.get("token_scores", {}).get(tok, {}).get("score", 0)
            print(f"    {tok:30s}  real={r_real:.4f}  random={r_rand:.4f}")

## Pass / fail summary

In [ ]:
print(f"{'='*60}")
print(f"SCORER EVALUATION SUMMARY: {type(scorer).__name__}")
print(f"{'='*60}\n")

checks = []

# 1. Clean separation: mean(real) > 3x mean(random)
ratio_random = real.mean() / max(results["random_dna"].mean(), 1e-8)
pass_separation = ratio_random > 3.0
checks.append(("Clean separation (real > 3x random)",
               pass_separation,
               f"ratio = {ratio_random:.1f}x"))

# 2. Non-overlapping IQRs with random
real_q1 = np.percentile(real, 25)
rand_q3 = np.percentile(results["random_dna"], 75)
pass_iqr = real_q1 > rand_q3
checks.append(("Non-overlapping IQR (real Q1 > random Q3)",
               pass_iqr,
               f"real Q1={real_q1:.4f}, random Q3={rand_q3:.4f}"))

# 3. AUROC vs random > 0.95
auroc_random = sep_df.loc[sep_df["negative"] == "random_dna", "auroc"].values[0]
pass_auroc = auroc_random > 0.95
checks.append(("AUROC vs random > 0.95",
               pass_auroc,
               f"AUROC = {auroc_random:.4f}"))

# 4. Graceful degradation: monotonic with truncation
checks.append(("Monotonic degradation with truncation",
               is_monotonic,
               f"means = {[f'{m:.3f}' for m in means]}"))

# 5. No false ceiling: real scores span a reasonable range
real_range = real.max() - real.min()
pass_range = real_range > 0.5
checks.append(("Score range > 0.5 (no false ceiling)",
               pass_range,
               f"range = {real_range:.4f}"))

# 6. Shuffled separation
ratio_shuffled = real.mean() / max(results["shuffled"].mean(), 1e-8)
pass_shuffled = ratio_shuffled > 2.0
checks.append(("Shuffled separation (real > 2x shuffled)",
               pass_shuffled,
               f"ratio = {ratio_shuffled:.1f}x"))

n_pass = 0
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    n_pass += int(passed)
    print(f"  [{status}] {name}")
    print(f"         {detail}")

print(f"\n{'─'*60}")
print(f"Result: {n_pass}/{len(checks)} checks passed")
if n_pass == len(checks):
    print("Scorer is ready for RL training.")
else:
    print("Scorer needs investigation before use in training.")